In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║           Speech Noise Reduction using PCA — Full Pipeline                   ║
║                                                                              ║
║  [1]  Download LibriSpeech  (kagglehub — no API key needed)                  ║
║  [2]  Download UrbanSound8K (kagglehub — no API key needed)                  ║
║  [3]  Preprocess: resample → mono → normalize                                ║
║  [4]  Mix signals at target SNR:  x = s + α·n                                ║
║  [5]  Frame signal into overlapping rectangular windows                      ║
║  [6]  Build data matrix  X = [x₁, x₂, …, xₙ]                                 ║
║  [7]  Mean-center:  X̃ = X − μ                                                ║
║  [8]  Covariance:   C = X̃X̃ᵀ / N                                              ║
║  [9]  Eigendecomposition:  C → V, Λ                                          ║
║  [10] Variance-threshold subspace selection (95%)                            ║
║  [11] Projection / denoising:  X̂ = Vₛ Vₛᵀ X̃ + μ                              ║
║  [12] Simple overlap-add reconstruction → waveform                           ║
║  [13] Evaluate: SNR, PESQ, Spectrograms, Waveforms                           ║
╚══════════════════════════════════════════════════════════════════════════════╝

Usage:
    pip install kagglehub librosa soundfile pesq matplotlib numpy
    python pca_speech_denoiser.py
"""


'\n╔══════════════════════════════════════════════════════════════════════════════╗\n║           Speech Noise Reduction using PCA — Full Pipeline                   ║\n║                                                                              ║\n║  [1]  Download LibriSpeech  (kagglehub — no API key needed)                  ║\n║  [2]  Download UrbanSound8K (kagglehub — no API key needed)                  ║\n║  [3]  Preprocess: resample → mono → normalize                                ║\n║  [4]  Mix signals at target SNR:  x = s + α·n                                ║\n║  [5]  Frame signal into overlapping rectangular windows                      ║\n║  [6]  Build data matrix  X = [x₁, x₂, …, xₙ]                                 ║\n║  [7]  Mean-center:  X̃ = X − μ                                                ║\n║  [8]  Covariance:   C = X̃X̃ᵀ / N                                              ║\n║  [9]  Eigendecomposition:  C → V, Λ                                          ║\n║  [10] Va

# Importing all the Required Libraries

In [ ]:
import sys, subprocess, warnings, random, csv
from pathlib import Path

warnings.filterwarnings("ignore")

# dependency check
_REQUIRED = ["kagglehub", "librosa", "soundfile", "pesq", "matplotlib", "numpy", "torch"]
print("[ ] Checking dependencies ...")
for _pkg in _REQUIRED:
    try:
        __import__(_pkg)
    except ImportError:
        print(f"    Installing {_pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", _pkg, "-q"],
                              stdout=subprocess.DEVNULL)
print("[OK] All dependencies ready.\n")

import numpy as np
import torch
import soundfile as sf
import librosa
import librosa.display
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pesq import pesq as pesq_score
import kagglehub

# GPU Check
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# default values
TARGET_SR       = 16_000
FRAME_SIZE      = 512
HOP_SIZE        = 256
VARIANCE_THRESH = 0.95
MIX_SNR_DB      = 0.0
NUM_SAMPLES     = 5
RANDOM_SEED     = 42
OUTPUT_DIR      = Path("output")

[ ] Checking dependencies ...
[OK] All dependencies ready.

Using device: cuda


# Function for Getting the Data from Kaggle

In [ ]:

def download_datasets() -> tuple[Path, Path]:
    """
    Download both datasets via kagglehub.
    Files are cached on disk after the first download.
    Returns (librispeech_root, urbansound8k_root).
    """
    print("=" * 60)
    print("[1] Downloading LibriSpeech from Kaggle ...")
    ls_root = Path(kagglehub.dataset_download("yasiashpot/librispeech"))
    print(f"    OK  LibriSpeech at: {ls_root}\n")

    print("[2] Downloading UrbanSound8K from Kaggle ...")
    us_root = Path(kagglehub.dataset_download("chrisfilo/urbansound8k"))
    print(f"    OK  UrbanSound8K at: {us_root}\n")

    return ls_root, us_root


Since the dataset on kaggle has audio files ,converting it to array first

In [ ]:
def load_audio(path: Path, target_sr: int = TARGET_SR) -> np.ndarray:
    """
    Load any audio file and return a clean 1-D float32 array.

    Applied operations:
      - Mono conversion  (librosa averages channels automatically)
      - Resampling to target_sr using a high-quality resampler
      - Peak normalization to [-1, 1]
    """
    y, sr = librosa.load(str(path), sr=None, mono=True) #ensures mono ie averages left and right channel,giving 1d signal
    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)#resamples signal ie converts it from discrete(by default stored as this in digital systems) to continous
    #scale the signal so that max of the peak is 1
    peak = np.max(np.abs(y))
    if peak > 1e-9:
        y = y / peak
    return y.astype(np.float32) #converts data to a numpy array
    #the signal that it loaded was already a binary file

#this picks the first file that has the appropriate correct extension ie from extensions
def pick_file(root: Path, extensions: tuple) -> Path:
    """Return the first audio file found recursively under root."""
    for p in sorted(root.rglob("*")):
        if p.suffix.lower() in extensions:
            return p
    raise FileNotFoundError(f"No audio file {extensions} found under {root}")


# Mixing the Audio with Noise to get input DATA

In [ ]:
def mix_at_snr(speech: np.ndarray, noise: np.ndarray,
               snr_db: float) -> tuple[np.ndarray, float]:
    """
    Scale noise so the mixture has the requested SNR, then add.

    Derivation
    ----------
        SNR_dB  = 10 * log10( Ps / Pn_scaled )
        alpha   = sqrt( Ps / (Pn * 10^(SNR_dB / 10)) ) we reach at this formula from the signal to noise ratio
        x       = s + alpha * n

    The mixture is peak-normalised to keep it in [-1, 1].
    Returns (mixture, alpha).
    """
    # noise signal must be atleast as long as speechsignal for addition so
    #repeat noise signal till it matches the required length
    if len(noise) < len(speech):
        noise = np.tile(noise, int(np.ceil(len(speech) / len(noise))))
    noise = noise[: len(speech)]
    #finding the intensity for finding the required alpha
    Ps    = float(np.mean(speech ** 2)) + 1e-12
    Pn    = float(np.mean(noise  ** 2)) + 1e-12
    alpha = float(np.sqrt(Ps / (Pn * 10.0 ** (snr_db / 10.0))))

    mixture = speech + alpha * noise
    peak    = np.max(np.abs(mixture))
    if peak > 1e-9:
        mixture = mixture / peak

    return mixture.astype(np.float32), alpha

Framing signals

In [ ]:
#split signal into multiple smaller frames
#so that we can analyze the smaller "windows" over time
#this is standard practice
def frame_signal(x: np.ndarray, frame_size: int, hop_size: int) -> np.ndarray:
    #Each column is one frame = one D-dimensional observation for PCA.
    L        = len(x)
    n_frames = 1 + (L - frame_size) // hop_size
    X        = np.zeros((frame_size, n_frames), dtype=np.float32)
    for i in range(n_frames):
        start    = i * hop_size
        X[:, i]  = x[start : start + frame_size]
    return X  # shape: (frame_size, number of frames)


# class for pca

In [ ]:
class PCASpeechDenoiser:
    """
    Mathematical pipeline (CUDA Optimized)
    """
    def __init__(self, variance_thresh: float = VARIANCE_THRESH, device=DEVICE):
        self.variance_thresh = variance_thresh
        self.device = device
        self.mean_ = None
        self.V_s = None
        self.eigenvalues_ = None
        self.k_ = None

    def fit(self, X_np: np.ndarray) -> "PCASpeechDenoiser":
        # Move data to GPU
        X = torch.from_numpy(X_np).to(self.device).to(torch.float64)
        D, N = X.shape

        # [7] Mean centering
        self.mean_ = X.mean(dim=1, keepdim=True)
        X_c = X - self.mean_

        # [8] Covariance matrix
        C = (X_c @ X_c.T) / N

        # [9] Eigendecomposition on GPU
        eigenvalues, eigenvectors = torch.linalg.eigh(C)

        # Sort descending
        idx = torch.argsort(eigenvalues, descending=True)
        eigenvalues = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]

        self.eigenvalues_ = eigenvalues.cpu().numpy()

        # [10] Variance threshold selection
        total = eigenvalues.sum() + 1e-12
        cumulative = torch.cumsum(eigenvalues, dim=0) / total

        # Find k
        k_idx = torch.where(cumulative >= self.variance_thresh)[0]
        self.k_ = int(k_idx[0].item() + 1) if len(k_idx) > 0 else D

        # Store the speech subspace basis on GPU
        self.V_s = eigenvectors[:, : self.k_].to(torch.float32)

        print(f"    PCA (CUDA) | D={D}, N={N}, k={self.k_} components")
        return self

    def denoise(self, X_np: np.ndarray) -> np.ndarray:
        # Process on GPU
        X = torch.from_numpy(X_np).to(self.device).to(torch.float32)
        X_c = X - self.mean_.to(torch.float32)

        # Project and reconstruct
        X_proj = self.V_s @ (self.V_s.T @ X_c)
        X_hat = X_proj + self.mean_.to(torch.float32)

        return X_hat.cpu().numpy()

    def scree_plot(self, save_path: Path, max_components: int = 64):
        n = min(max_components, len(self.eigenvalues_))
        total = self.eigenvalues_.sum() + 1e-12
        indiv = self.eigenvalues_[:n] / total * 100
        cumul = np.cumsum(self.eigenvalues_[:n]) / total * 100
        x = np.arange(1, n + 1)

        fig, ax1 = plt.subplots(figsize=(9, 4))
        ax2 = ax1.twinx()
        ax1.bar(x, indiv, color="#2196F3", alpha=0.6)
        ax2.plot(x, cumul, "o-", color="#F44336", ms=3)
        ax2.axhline(self.variance_thresh * 100, ls="--", color="#4CAF50")
        plt.savefig(save_path, dpi=150)
        plt.close()

# Function for the reverse of the framing when we want to recreate the original signal

In [ ]:
#example for this :
'''
Frame 0: [a b c d]
Frame 1:     [e f g h]
Frame 2:         [i j k l]
'''
def overlap_add(frames: np.ndarray, hop_size: int,
                signal_length: int) -> np.ndarray:
    """
    Reconstruct a waveform by summing overlapping frames and dividing by the
    number of frames that contributed to each output sample (simple average).
        output[t]  = sum of all frame samples that cover position t
        count[t]   = number of frames that cover position t
        result[t]  = output[t] / count[t]
    """
    frame_size, n_frames = frames.shape
    output = np.zeros(signal_length, dtype=np.float64)
    count  = np.zeros(signal_length, dtype=np.float64)

    for i in range(n_frames):
        start = i * hop_size
        end   = start + frame_size
        if end > signal_length:          # safety guard
            break
        output[start:end] += frames[:, i].astype(np.float64)
        count[start:end]  += 1.0

    count = np.where(count < 1.0, 1.0, count)   # avoid divide-by-zero
    return (output / count).astype(np.float32)


In [ ]:
def list_audio_files(root: Path, extensions: tuple[str, ...]) -> list[Path]:
    """Return all matching audio files recursively under root."""
    files = [p for p in sorted(root.rglob("*")) if p.suffix.lower() in extensions]
    if not files:
        raise FileNotFoundError(f"No audio files {extensions} found under {root}")
    return files


def sample_without_replacement(files: list[Path], k: int, rng: random.Random) -> list[Path]:
    if k > len(files):
        raise ValueError(f"Requested {k}, but only {len(files)} files available")
    return rng.sample(files, k)


def get_user_config() -> dict:
    def ask(prompt: str, default, cast, validator=None):
        while True:
            try:
                raw = input(f"{prompt} [default={default}]: ").strip()
            except Exception:
                # Some notebook frontends may not support input requests.
                print(f"Input not available for '{prompt}'. Using default={default}.")
                return default

            try:
                value = cast(raw) if raw else default
            except Exception:
                print("Invalid value type. Try again.")
                continue

            if validator and not validator(value):
                print("Value out of range. Try again.")
                continue

            return value

    config = {}
    config["TARGET_SR"] = ask("Sampling rate (Hz)", TARGET_SR, int, lambda v: v > 0)
    config["FRAME_SIZE"] = ask("Frame size", FRAME_SIZE, int, lambda v: v > 1)
    config["HOP_SIZE"] = ask("Hop size", HOP_SIZE, int, lambda v: v > 0)
    config["VARIANCE_THRESH"] = ask("Variance threshold (0-1)", VARIANCE_THRESH, float, lambda v: 0.0 < v <= 1.0)
    config["MIX_SNR_DB"] = ask("Mix SNR (dB)", MIX_SNR_DB, float)
    config["NUM_SAMPLES"] = ask("Number of speech files to evaluate", NUM_SAMPLES, int, lambda v: v > 0)
    config["RANDOM_SEED"] = ask("Random seed", RANDOM_SEED, int)
    config["OUTPUT_DIR"] = ask("Output directory", str(OUTPUT_DIR), str, lambda v: len(v) > 0)

    if config["HOP_SIZE"] > config["FRAME_SIZE"]:
        raise ValueError("HOP_SIZE must be <= FRAME_SIZE")

    return config


def compute_snr(clean: np.ndarray, degraded: np.ndarray) -> float:
    """Signal-to-Noise Ratio in dB."""
    Ps = float(np.mean(clean ** 2)) + 1e-12
    Pn = float(np.mean((clean - degraded) ** 2)) + 1e-12
    return 10.0 * np.log10(Ps / Pn)


def compute_pesq(clean: np.ndarray, degraded: np.ndarray, sr: int) -> float:
    """Wideband PESQ score in [-0.5, 4.5] for sr=16000."""
    n = min(len(clean), len(degraded))
    try:
        return float(pesq_score(sr, clean[:n], degraded[:n], "wb"))
    except Exception as e:
        print(f"    WARNING: PESQ failed: {e}")
        return float("nan")


def plot_spectrograms(
    speech: np.ndarray,
    mixture: np.ndarray,
    denoised: np.ndarray,
    sr: int,
    mix_snr_db: float,
    save_path: Path,
):
    """Three-panel STFT magnitude spectrogram: clean | noisy | denoised."""
    hop = 128
    n_fft = 512
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
    signals = [speech, mixture, denoised]
    titles = [
        "Clean Speech",
        f"Noisy Mixture (SNR={mix_snr_db} dB)",
        "PCA Denoised",
    ]
    cmaps = ["Blues_r", "Reds_r", "Greens_r"]
    img = None
    for ax, sig, title, cmap in zip(axes, signals, titles, cmaps):
        D = librosa.stft(sig, n_fft=n_fft, hop_length=hop)
        S = librosa.amplitude_to_db(np.abs(D), ref=np.max)
        img = librosa.display.specshow(
            S, sr=sr, hop_length=hop, x_axis="time", y_axis="hz", ax=ax, cmap=cmap
        )
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.set_xlabel("Time (s)")
    axes[0].set_ylabel("Frequency (Hz)")
    fig.colorbar(img, ax=axes.tolist(), format="%+2.0f dB", shrink=0.8)
    plt.suptitle("Spectrogram Comparison", fontsize=14, y=1.03)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"    OK  Spectrograms  -> {save_path}")


def plot_waveforms(
    speech: np.ndarray,
    mixture: np.ndarray,
    denoised: np.ndarray,
    sr: int,
    mix_snr_db: float,
    save_path: Path,
):
    """Three-panel time-domain amplitude plot."""
    t = np.arange(len(speech)) / sr
    signals = [speech, mixture, denoised]
    titles = [
        "Clean Speech",
        f"Noisy Mixture (SNR={mix_snr_db} dB)",
        "PCA Denoised",
    ]
    colors = ["#2196F3", "#F44336", "#4CAF50"]

    fig, axes = plt.subplots(3, 1, figsize=(13, 6), sharex=True)
    for ax, sig, title, col in zip(axes, signals, titles, colors):
        ax.plot(t[: len(sig)], sig, color=col, lw=0.5, alpha=0.85)
        ax.set_ylabel("Amplitude", fontsize=9)
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.set_ylim(-1.15, 1.15)
        ax.grid(True, alpha=0.25)
    axes[-1].set_xlabel("Time (s)", fontsize=10)
    plt.suptitle("Waveform Comparison", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"    OK  Waveforms     -> {save_path}")


def print_results(snr_n: float, snr_d: float, pesq_n: float, pesq_d: float):
    W = 52
    print("\n" + "=" * W)
    print(f"  {'EVALUATION RESULTS':^{W-4}}")
    print("=" * W)
    print(f"  {'Metric':<26} {'Noisy':>8}  {'Denoised':>8}")
    print("-" * W)
    print(f"  {'SNR (dB)':<26} {snr_n:>8.2f}  {snr_d:>8.2f}")
    print(f"  {'PESQ  (MOS-LQO, 0-4.5)':<26} {pesq_n:>8.3f}  {pesq_d:>8.3f}")
    print("-" * W)
    print(f"  {'delta SNR':<26} {snr_d - snr_n:>+17.2f} dB")
    print(f"  {'delta PESQ':<26} {pesq_d - pesq_n:>+17.3f}")
    print("=" * W)


def save_metrics_csv(rows: list[dict], csv_path: Path):
    fields = [
        "case",
        "speech_file",
        "noise_file",
        "snr_noisy",
        "snr_denoised",
        "delta_snr",
        "pesq_noisy",
        "pesq_denoised",
        "delta_pesq",
    ]
    with csv_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def summarize_metrics(rows: list[dict]):
    snr_noisy = np.array([r["snr_noisy"] for r in rows], dtype=np.float64)
    snr_denoised = np.array([r["snr_denoised"] for r in rows], dtype=np.float64)
    pesq_noisy = np.array([r["pesq_noisy"] for r in rows], dtype=np.float64)
    pesq_denoised = np.array([r["pesq_denoised"] for r in rows], dtype=np.float64)

    def mean_std(x: np.ndarray) -> tuple[float, float]:
        valid = x[~np.isnan(x)]
        if len(valid) == 0:
            return float("nan"), float("nan")
        return float(np.mean(valid)), float(np.std(valid))

    snr_n_mu, snr_n_std = mean_std(snr_noisy)
    snr_d_mu, snr_d_std = mean_std(snr_denoised)
    pesq_n_mu, pesq_n_std = mean_std(pesq_noisy)
    pesq_d_mu, pesq_d_std = mean_std(pesq_denoised)

    print("\n" + "=" * 72)
    print("AGGREGATE METRICS ACROSS ALL CASES")
    print("=" * 72)
    print(f"SNR  noisy    : {snr_n_mu:8.3f} +/- {snr_n_std:8.3f} dB")
    print(f"SNR  denoised : {snr_d_mu:8.3f} +/- {snr_d_std:8.3f} dB")
    print(f"dSNR          : {(snr_d_mu - snr_n_mu):8.3f} dB")
    print(f"PESQ noisy    : {pesq_n_mu:8.3f} +/- {pesq_n_std:8.3f}")
    print(f"PESQ denoised : {pesq_d_mu:8.3f} +/- {pesq_d_std:8.3f}")
    print(f"dPESQ         : {(pesq_d_mu - pesq_n_mu):8.3f}")
    print("=" * 72)


def main():
    config = get_user_config()
    target_sr = int(config["TARGET_SR"])
    frame_size = int(config["FRAME_SIZE"])
    hop_size = int(config["HOP_SIZE"])
    variance_thresh = float(config["VARIANCE_THRESH"])
    mix_snr_db = float(config["MIX_SNR_DB"])
    requested_k = int(config["NUM_SAMPLES"])
    rng = random.Random(int(config["RANDOM_SEED"]))
    output_dir = Path(config["OUTPUT_DIR"])
    output_dir.mkdir(parents=True, exist_ok=True)

    print("  PCA-Based Speech Noise Reduction")
    print("=" * 60 + "\n")

    ls_root, us_root = download_datasets()

    audio_exts = (".flac", ".wav", ".mp3", ".ogg")
    speech_files = list_audio_files(ls_root, audio_exts)
    noise_files = list_audio_files(us_root, audio_exts)

    max_pairs = min(len(speech_files), len(noise_files))
    if requested_k > max_pairs:
        print(f"Requested {requested_k} files, but only {max_pairs} non-repeating pairs are possible.")
        print(f"Proceeding with NUM_SAMPLES={max_pairs} due to no-replacement sampling.")
        requested_k = max_pairs

    selected_speech = sample_without_replacement(speech_files, requested_k, rng)
    selected_noise = sample_without_replacement(noise_files, requested_k, rng)

    print(f"Selected {requested_k} speech files and {requested_k} noise files.")
    print("Building training matrix from all selected noisy samples...")

    cases = []
    train_frames = []
    for idx, (speech_path, noise_path) in enumerate(zip(selected_speech, selected_noise), start=1):
        speech = load_audio(speech_path, target_sr=target_sr)
        noise = load_audio(noise_path, target_sr=target_sr)
        mixture, alpha = mix_at_snr(speech, noise, snr_db=mix_snr_db)

        L = min(len(speech), len(mixture))
        speech = speech[:L]
        mixture = mixture[:L]

        if L < frame_size:
            print(f"Case {idx}: skipped (signal shorter than FRAME_SIZE)")
            continue

        X = frame_signal(mixture, frame_size, hop_size)
        train_frames.append(X)
        cases.append({
            "case": idx,
            "speech_path": speech_path,
            "noise_path": noise_path,
            "speech": speech,
            "mixture": mixture,
            "X": X,
            "L": L,
            "alpha": alpha,
        })
        print(f"Case {idx}: speech={speech_path.name}, noise={noise_path.name}, frames={X.shape[1]}")

    if not cases:
        raise RuntimeError("No valid cases found. Reduce FRAME_SIZE or check datasets.")

    X_train = np.concatenate(train_frames, axis=1)
    print(f"Combined training matrix shape: {X_train.shape}")

    denoiser = PCASpeechDenoiser(variance_thresh=variance_thresh)
    denoiser.fit(X_train)

    rows = []
    representative = None

    for c in cases:
        X_hat = denoiser.denoise(c["X"])
        denoised = overlap_add(X_hat, hop_size, signal_length=c["L"])
        L = min(len(c["speech"]), len(c["mixture"]), len(denoised))
        speech = c["speech"][:L]
        mixture = c["mixture"][:L]
        denoised = denoised[:L]

        snr_n = compute_snr(speech, mixture)
        snr_d = compute_snr(speech, denoised)
        pesq_n = compute_pesq(speech, mixture, sr=target_sr)
        pesq_d = compute_pesq(speech, denoised, sr=target_sr)

        print(f"\nCase {c['case']} results:")
        print_results(snr_n, snr_d, pesq_n, pesq_d)

        row = {
            "case": c["case"],
            "speech_file": c["speech_path"].name,
            "noise_file": c["noise_path"].name,
            "snr_noisy": snr_n,
            "snr_denoised": snr_d,
            "delta_snr": snr_d - snr_n,
            "pesq_noisy": pesq_n,
            "pesq_denoised": pesq_d,
            "delta_pesq": pesq_d - pesq_n,
        }
        rows.append(row)

        if representative is None:
            representative = {
                "case": c["case"],
                "speech": speech,
                "mixture": mixture,
                "denoised": denoised,
            }

    if representative is None:
        raise RuntimeError("No representative sample available for plotting.")

    rep_dir = output_dir / "representative_case"
    rep_dir.mkdir(parents=True, exist_ok=True)
    sf.write(rep_dir / "01_clean_speech.wav", representative["speech"], target_sr)
    sf.write(rep_dir / "02_noisy_mixture.wav", representative["mixture"], target_sr)
    sf.write(rep_dir / "03_pca_denoised.wav", representative["denoised"], target_sr)

    denoiser.scree_plot(output_dir / "scree_plot.png")
    plot_spectrograms(
        representative["speech"],
        representative["mixture"],
        representative["denoised"],
        target_sr,
        mix_snr_db,
        output_dir / "representative_spectrograms.png",
    )
    plot_waveforms(
        representative["speech"],
        representative["mixture"],
        representative["denoised"],
        target_sr,
        mix_snr_db,
        output_dir / "representative_waveforms.png",
    )

    save_metrics_csv(rows, output_dir / "metrics_summary.csv")
    summarize_metrics(rows)

    print("\nDone. Outputs saved in:")
    print(f"  {output_dir}")
    print(f"  {rep_dir}")
    print("  metrics_summary.csv")
    print("  representative_spectrograms.png")
    print("  representative_waveforms.png")
    print("  scree_plot.png")


if __name__ == "__main__":
    main()

Sampling rate (Hz) [default=16000]: 
Frame size [default=512]: 
Hop size [default=256]: 
Variance threshold (0-1) [default=0.95]: 
Mix SNR (dB) [default=0.0]: 60
Number of speech files to evaluate [default=5]: 
Random seed [default=42]: 
Output directory [default=output]: 
  PCA-Based Speech Noise Reduction

[1] Downloading LibriSpeech from Kaggle ...
Using Colab cache for faster access to the 'librispeech' dataset.
    OK  LibriSpeech at: /kaggle/input/librispeech

[2] Downloading UrbanSound8K from Kaggle ...
Using Colab cache for faster access to the 'urbansound8k' dataset.
    OK  UrbanSound8K at: /kaggle/input/urbansound8k

Selected 5 speech files and 5 noise files.
Building training matrix from all selected noisy samples...
Case 1: speech=8463-294825-0008.flac, noise=144007-5-1-9.wav, frames=247
Case 2: speech=5895-34629-0033.flac, noise=203929-7-6-3.wav, frames=482
Case 3: speech=1993-147149-0002.wav, noise=93567-8-0-7.wav, frames=359
Case 4: speech=672-122797-0018.flac, noise=20